# Tokenization

Tokenization is how an LLM converts text into numbers.
Since neural networks work on numbers, not characters, we first break text into small units called tokens.

In simple tokenization, a word = a token.
In modern LLMs, words are broken into subwords, so the model can handle new or rare words without needing them in the vocabulary.

_Text → Tokenization → Embeddings → Transformer (attention & layers) → Logits → Decoding → Detokenization → Output text_

## Example 1 — A very simple tokenizer (splits on spaces & punctuation)

The core idea: text → tokens → numbers.

In [ ]:
import re

text = "Large Language Models are amazing. Large Language Models are powerful."

# ---- Simple tokenizer: split on spaces & punctuation ----
def simple_tokenize(sentence):
    # Keep only letters, numbers and spaces
    cleaned = re.sub(r"[^a-zA-Z0-9\s]", "", sentence) 
    # The regex replaces every character in sentence that is not a letter, digit, or whitespace with an empty string (i.e., it strips punctuation and other symbols).
    
    # Split by whitespace
    return cleaned.split()

tokens = simple_tokenize(text)

print("Original text:", text)
print("Tokens:", tokens)

# ---- Create a tiny vocabulary ----
vocab = {word: i for i, word in enumerate(sorted(set(tokens)))}
print("Vocabulary:", vocab)

# ---- Convert tokens → ids ----
token_ids = [vocab[word] for word in tokens]
print("Token IDs:", token_ids)

This works for very simple applications, but breaks for:

- Unknown words
- Typos
- Other languages
- Emojis
- Doesn’t handle subwords


## Example 2 — A modern tokenizer (BPE / WordPiece) using Hugging Face

This is what real LLMs use: subword tokenization
(e.g., “playing” → “play” + “ing”)

In [ ]:
from transformers import AutoTokenizer

text = "Large Language Models are amazing. Large Language Models are powerful."

# ---- Load a modern tokenizer ----
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# ---- Encode: text → token IDs ----
encoded = tokenizer.encode(text)
print("Token IDs:", encoded)

# ---- Decode: token IDs → text ----
decoded = tokenizer.decode(encoded)
print("Decoded Text:", decoded)

# ---- See the subword tokens ----
tokens = tokenizer.convert_ids_to_tokens(encoded)
print("Tokens:", tokens)

Modern LLM tokenizers:

- Break words into subwords / pieces
- Handle new / rare words gracefully
- Work across languages, symbols, emojis
- Keep vocabulary size manageable (≈ 30k–100k)


## How to create efficient Tokenizers:

### What is BPE (Byte-Pair Encoding)?

Byte-Pair Encoding is a method for turning text into subword tokens.
The idea is simple:

- Start with individual characters as tokens
- Count which pairs of characters occur most often
- Merge the most frequent pair into a new token
- Repeat… until you have the vocabulary size you want

So the tokenizer learns common chunks like:

- "t"
- "h"
- "th"
- "the"
- "ing"
- "tion"


So, as a result

- Common words → become single tokens (the, cat, model)
- Rare/long words → get broken into pieces (amaz + ing, micro + scope)
- No word is ever “unknown”, because characters still exist as a fallback

Modern LLMs mostly use Byte-Level BPE, meaning it works at raw byte level, so it can tokenize:
- any language
- emojis 😀
- symbols
- typos

Tokenization is usually learned separately — before model training. The usual pipeline is:

- Take a huge text dataset
- Run BPE (or WordPiece / Unigram LM, etc.)
- Build a vocabulary (e.g., 32k–100k tokens)
- Save the tokenizer

This tokenizer stays frozen during the next training phases of LLMs. 
